# Terra Analog Atlas — inspect the finished MVP

Open this notebook from the extracted project directory. In Colab, first upload and extract the project ZIP into `/content/Terra_Analog_Atlas`. All features below come from actual source rasters. No training or GPU is required.

The web app is easiest to run locally: `python run.py`. This notebook inspects the scientific snapshot and uses the same Python ranking module.

In [ ]:
from pathlib import Path
import json, sys
ROOT = Path.cwd()
if not (ROOT / 'dist/data/atlas.json').exists():
    ROOT = Path('/content/Terra_Analog_Atlas')
assert (ROOT / 'dist/data/atlas.json').exists(), 'Extract the full project ZIP and set ROOT to that folder.'
sys.path.insert(0, str(ROOT))
data = json.loads((ROOT / 'dist/data/atlas.json').read_text())
print(len(data['candidates']), 'real Earth patches')
print([t['name'] for t in data['targets']])

## Install the project dependencies if needed

The following cell installs the project into the current notebook environment. It does not download raw rasters or require credentials.

In [ ]:
import subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', str(ROOT)])

In [ ]:
from analog_atlas.ranking import rank
selected_target = next(t for t in data['targets'] if t['id'] == 'jezero')
weights = {'slope': 1, 'relief': 1, 'roughness': 1}
ranked = rank(data['candidates'], selected_target, data['scaler'], weights)
by_id = {p['id']: p for p in data['candidates']}
for i, row in enumerate(ranked[:5], 1):
    print(i, by_id[row['id']]['name'], round(row['distance'], 4))

## Inspect evidence, not a percentage match

In [ ]:
best = by_id[ranked[0]['id']]
for key in data['feature_keys']:
    print(key, 'Earth:', round(best['features'][key], 3), 'Target:', round(selected_target['features'][key], 3))
print(data['limitations'])

In [ ]:
from IPython.display import display, Image
display(Image(filename=str(ROOT / 'dist' / best['image']), width=300))
display(Image(filename=str(ROOT / 'dist' / selected_target['image']), width=300))
print('Independent colour scales; elevations, not optical surface photographs.')

## Reproduce validation

These checks test implementation and sensitivity. They do not establish field relevance or accuracy.

In [ ]:
subprocess.run([sys.executable, '-m', 'pytest', '-q'], cwd=ROOT, check=True)
print(json.dumps(data['validation'], indent=2))